### Data Ingestion to vector DB pipeline


In [12]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter  
from pathlib import Path

In [15]:
### Read all the PDF in the directory
def process_all_pdf(pdf_directory):
    all_documents=[]
    pdf_dir= Path(pdf_directory)

    pdf_files=list(pdf_dir.glob("**/*.pdf")) 

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source'] = str(pdf_file.name)
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}")
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdf("../data/pdf")


        

Found 4 PDF files to process.
Processing file: BA UNIT 4.pdf
Loaded 32 pages from BA UNIT 4.pdf
Processing file: BA_UNIT 5 (1).pdf
Loaded 24 pages from BA_UNIT 5 (1).pdf
Processing file: Marketing analytics overview.pdf
Error processing Marketing analytics overview.pdf: 'bbox'
Processing file: Unit 3 B.pdf
Loaded 31 pages from Unit 3 B.pdf
Total documents loaded: 87


In [16]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m120 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'EBA UNIT 4', 'source': 'BA UNIT 4.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='BAUNIT-4\nMarketAnalytics, Modelsandmetrics, MarketInsight–Marketdatasources, sizing, PESTLEtrendanalysis, andporter5forcesanalysis–MarketbasketAnalysis, TextAnalytics, SpreadsheetModelling–SalesAnalytics: ECommercesalesmode, salesmetricsprofitabilitymetricsandsupportmetrics.\nWhatisMarketingAnalytics?Marketing analytics is thepracticeof using data toevaluatetheeffectiveness andsuccess ofmarketingactivities. Marketing analytics allows youtogather deeper consumer insights, optimizeyour marketingobjectives, andgetabetterreturnoninvestment.\nMarketing analytics benefits both marketers and consumers. This analysis allows marketers toachievehigher ROI onmarketing investments by understanding what is successful indriving either conversions,brand awaren

In [17]:
####chunks


def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs= text_splitter.split_documents(documents)
    print(f"Split{len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print("Sample chunk content:")
        print(f"Content:{split_docs[0].page_content[:200]}") 
         # Print first 200 characters of the first chunk
        print(f"Metadata:{split_docs[0].metadata}")  # Print metadata of the first chunk

    return split_docs


In [19]:
chunk=split_documents(all_pdf_documents)
chunk

Split87 documents into 250 chunks.
Sample chunk content:
Content:BAUNIT-4
MarketAnalytics, Modelsandmetrics, MarketInsight–Marketdatasources, sizing, PESTLEtrendanalysis, andporter5forcesanalysis–MarketbasketAnalysis, TextAnalytics, SpreadsheetModelling–SalesAnalyt
Metadata:{'producer': 'Skia/PDF m120 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'EBA UNIT 4', 'source': 'BA UNIT 4.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m120 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'EBA UNIT 4', 'source': 'BA UNIT 4.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='BAUNIT-4\nMarketAnalytics, Modelsandmetrics, MarketInsight–Marketdatasources, sizing, PESTLEtrendanalysis, andporter5forcesanalysis–MarketbasketAnalysis, TextAnalytics, SpreadsheetModelling–SalesAnalytics: ECommercesalesmode, salesmetricsprofitabilitymetricsandsupportmetrics.\nWhatisMarketingAnalytics?Marketing analytics is thepracticeof using data toevaluatetheeffectiveness andsuccess ofmarketingactivities. Marketing analytics allows youtogather deeper consumer insights, optimizeyour marketingobjectives, andgetabetterreturnoninvestment.\nMarketing analytics benefits both marketers and consumers. This analysis allows marketers toachievehigher ROI onmarketing investments by understanding what is successful indriving either conversions,brand awaren

In [23]:
### EMBEDDING AND VECTOR STORE --- IGNORE ---
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [24]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Projectsllm\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\niico\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP

Model loaded successfully. Embedding dimension: 384


In [25]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [27]:
chunk

[Document(metadata={'producer': 'Skia/PDF m120 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'EBA UNIT 4', 'source': 'BA UNIT 4.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='BAUNIT-4\nMarketAnalytics, Modelsandmetrics, MarketInsight–Marketdatasources, sizing, PESTLEtrendanalysis, andporter5forcesanalysis–MarketbasketAnalysis, TextAnalytics, SpreadsheetModelling–SalesAnalytics: ECommercesalesmode, salesmetricsprofitabilitymetricsandsupportmetrics.\nWhatisMarketingAnalytics?Marketing analytics is thepracticeof using data toevaluatetheeffectiveness andsuccess ofmarketingactivities. Marketing analytics allows youtogather deeper consumer insights, optimizeyour marketingobjectives, andgetabetterreturnoninvestment.\nMarketing analytics benefits both marketers and consumers. This analysis allows marketers toachievehigher ROI onmarketing investments by understanding what is successful indriving either conversions,brand awaren

In [29]:
texts=[doc.page_content for doc in chunk]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunk,embeddings)

Generating embeddings for 250 texts...


Batches: 100%|██████████| 8/8 [00:13<00:00,  1.67s/it]


Generated embeddings with shape: (250, 384)
Adding 250 documents to vector store...
Successfully added 250 documents to vector store
Total documents in collection: 250


In [30]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [31]:
rag_retriever

In [32]:
rag_retriever.retrieve("What is Marketing Analytics?")

Retrieving documents for query: 'What is Marketing Analytics?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_a758f5f0_0',
  'content': 'BAUNIT-4\nMarketAnalytics, Modelsandmetrics, MarketInsight–Marketdatasources, sizing, PESTLEtrendanalysis, andporter5forcesanalysis–MarketbasketAnalysis, TextAnalytics, SpreadsheetModelling–SalesAnalytics: ECommercesalesmode, salesmetricsprofitabilitymetricsandsupportmetrics.\nWhatisMarketingAnalytics?Marketing analytics is thepracticeof using data toevaluatetheeffectiveness andsuccess ofmarketingactivities. Marketing analytics allows youtogather deeper consumer insights, optimizeyour marketingobjectives, andgetabetterreturnoninvestment.\nMarketing analytics benefits both marketers and consumers. This analysis allows marketers toachievehigher ROI onmarketing investments by understanding what is successful indriving either conversions,brand awareness, or both. Analytics also ensures that consumers see a greater number of targeted,personalizedads that speak totheir specific needsandinterests, ratherthanmasscommunicationsthattendtoannoy.',
  'metada

In [33]:
rag_retriever.retrieve("What is PESTLE Analysis")

Retrieving documents for query: 'What is PESTLE Analysis'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.17it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_daec3df6_22',
  'content': 'What is PESTLEAnalysis? PESTLEanalysis, whichis sometimes referredtoas PESTanalysis, is aconceptinmarketing principles. Moreover, this concept is usedas a tool bycompaniestotracktheenvironmentthey’reoperating inor areplanning tolauncha newproject/product/service, etc. So, let’sfindoutwhattheselettersstandforfirst.\nPESTLEis a mnemonic whichinits expandedformdenotes Pfor Political, EforEconomic, SforSocial, Tfor Technological, L for Legal, and E for Environmental. It gives a bird’s eye view of the wholeenvironment from many different angles that one wants to check and keep a track of whilecontemplating a certainidea/plan. ThePESTELframeworkhasundergonecertainalterations, asgurusofMarketinghaveaddedcertainthingslikeanEforEthicstoinstill theelementofdemographicswhileutilizingtheframeworkwhileresearchingthemarket.\nAskingtheRightQuestionsTherearecertainquestionsthatoneneedstoaskwhileconductingthisanalysis, whichgivesthemanideaofwhatthingstokeepinmin

In [50]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

None


In [58]:
from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate
from langchain.schema import HumanMessage, SystemMessage


ModuleNotFoundError: No module named 'langchain.prompts'

In [59]:
import langchain
import langchain_groq


In [61]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [64]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = "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"
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [65]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [66]:
rag_retriever.retrieve("What is Pestle Analysis?")

Retrieving documents for query: 'What is Pestle Analysis?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.74it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_daec3df6_22',
  'content': 'What is PESTLEAnalysis? PESTLEanalysis, whichis sometimes referredtoas PESTanalysis, is aconceptinmarketing principles. Moreover, this concept is usedas a tool bycompaniestotracktheenvironmentthey’reoperating inor areplanning tolauncha newproject/product/service, etc. So, let’sfindoutwhattheselettersstandforfirst.\nPESTLEis a mnemonic whichinits expandedformdenotes Pfor Political, EforEconomic, SforSocial, Tfor Technological, L for Legal, and E for Environmental. It gives a bird’s eye view of the wholeenvironment from many different angles that one wants to check and keep a track of whilecontemplating a certainidea/plan. ThePESTELframeworkhasundergonecertainalterations, asgurusofMarketinghaveaddedcertainthingslikeanEforEthicstoinstill theelementofdemographicswhileutilizingtheframeworkwhileresearchingthemarket.\nAskingtheRightQuestionsTherearecertainquestionsthatoneneedstoaskwhileconductingthisanalysis, whichgivesthemanideaofwhatthingstokeepinmin

In [71]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = "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"

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [72]:
answer=rag_simple("What is pestle analysis?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is pestle analysis?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.77it/s]


Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
PESTLE analysis is a marketing concept used to track the environment a company operates in, standing for Political, Economic, Social, Technological, Legal, and Environmental factors.


In [77]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Marketing Analytics?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:500])

Retrieving documents for query: 'What is Marketing Analytics?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Marketing analytics is the practice of using data to evaluate the effectiveness and success of marketing activities.
Sources: [{'source': 'BA UNIT 4.pdf', 'page': 0, 'score': 0.39727628231048584, 'preview': 'BAUNIT-4\nMarketAnalytics, Modelsandmetrics, MarketInsight–Marketdatasources, sizing, PESTLEtrendanalysis, andporter5forcesanalysis–MarketbasketAnalysis, TextAnalytics, SpreadsheetModelling–SalesAnalytics: ECommercesalesmode, salesmetricsprofitabilitymetricsandsupportmetrics.\nWhatisMarketingAnalytics...'}, {'source': 'BA UNIT 4.pdf', 'page': 2, 'score': 0.3567754030227661, 'preview': '4. Actonwhatyoulearn\nUsing data is one of thegreatest challenges facing marketing professionals thesedays. There’s just so.Much. Data! That’s why Step 1 is so important: If you know that what you’re currently doing isn’thelpingyoureachyourgoals, thenyouknowit’stimetotestanditerate.\nApplied holistica...'}, {'source': 'Unit 3 B.pdf', 'page': 1, 'score': 0.303127646446228, 'preview': 'involved

In [79]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is pestle Analysis?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What is pestle Analysis?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
What is PESTLEAnalysis? PESTLEanalysis, whichis sometimes referredtoas PESTanalysis, is aconceptinmarketing principles. Moreover, this concept is usedas a tool bycompaniestotracktheenvironmentthey’reoperating inor areplanning tolauncha newproject/prod

uct/service, etc. So, let’sfindoutwhattheselettersstandforfirst.
PESTLEis a mnemonic whichinits expandedformdenotes Pfor Political, EforEconomic, SforSocial, Tfor Technological, L for Legal, and E for Environmental. It gives a bird’s eye view of the wholeenvironment from many different angles that one wants to check and keep a track of whilecontemplating a certainidea/plan. ThePESTELframeworkhasundergonecertainalterations, asgurusofMarketinghaveaddedcertainthingslikeanEforEthicstoinstill theelementofdemographicswhileutilizingtheframeworkwhileresearchingthemarket.
AskingtheRightQuestionsTherearecertainquestionsthatoneneedstoaskwhileconductingthisanalysis, whichgivesthemanideaofwhatthingstokeepinmind. Theyare:

Question: What is pestle Analysis?

Answer:

Final Answer: PESTLE Analysis is a marketing concept used to track the environment a company operates in, standing for Political, Economic, Social, Technological, Legal, and Environmental factors.

Citations:
[1] BA UNIT 4.pdf (page 9)
